# Fairness processing methods

## Requirements

In [1]:
import numpy as np
import pandas as pd 

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split 

from aif360.datasets import GermanDataset

import tensorflow as tf

from fairlearn.reductions import Moment # necessary for ExponentiatedGradientReduction

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens
from PyFairnessAI.metrics import statistical_parity_difference
from PyFairnessAI.preprocessing import ReweighingMetaEstimator
from PyFairnessAI.inprocessing import (AdversarialDebiasingEstimator, ExponentiatedGradientReductionEstimator, 
                                       GridSearchReductionEstimator, Moment)
from PyFairnessAI.postprocessing import CalibratedEqualizedOdds, RejectOptionClassifier, PostProcessingMeta

pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
pip install 'aif360[FairAdapt]'
pip install 'aif360[LFR]'


## Initial elements

### Data

In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
# The sensitive variable/s must index the df in order to avoid problems with some aif360 processors (PostProcessingMetaEstimator)
german_data_pd.index = german_data_pd[sens_variable_name] 
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
age,,,,,,,,,,,,,,,,,,,,,
1.0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
0.0,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [4]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [5]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

In [36]:
Y_train

age
1.0    0.0
1.0    0.0
1.0    1.0
1.0    1.0
1.0    0.0
      ... 
1.0    1.0
1.0    1.0
1.0    1.0
1.0    1.0
1.0    1.0
Name: credit, Length: 850, dtype: float64

## Pre-processing

In [7]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
log_reg_estimator.fit(X_train, Y_train)
Y_test_hat = log_reg_estimator.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

In [8]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.32539682539682535

In [9]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

### Reweighing

In [10]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
reweighing_log_reg_estimator = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)
reweighing_log_reg_estimator.fit(X=X_train, y=Y_train)
Y_test_hat_reweighing = reweighing_log_reg_estimator.predict(X_test)

In [11]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat_reweighing, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

In [12]:
balanced_accuracy_score(y_pred=Y_test_hat_reweighing, y_true=Y_test)

0.6396825396825396

## In-processing

### Adversarial Debiasing

In [13]:
# Set up TensorFlow session (required by AdversarialDebiasingEstimator)
tf_session = tf.compat.v1.Session
# disable_eager_execution is required as well by TensorFlow
tf.compat.v1.disable_eager_execution()

In [14]:
adversarial_debiasing = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

In [15]:
adversarial_debiasing.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

AdversarialDebiasingEstimator(prot_attr='age', random_state=123, verbose=True)

In [16]:
Y_test_hat = adversarial_debiasing.predict(X_test)
Y_test_hat

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [17]:
adversarial_debiasing.predict_proba(X_test)

array([[4.35757606e-02, 9.56424239e-01],
       [3.26964053e-01, 6.73035947e-01],
       [1.65164924e-02, 9.83483508e-01],
       [3.68154000e-04, 9.99631846e-01],
       [8.42234403e-02, 9.15776560e-01],
       [2.01420627e-02, 9.79857937e-01],
       [1.17699089e-02, 9.88230091e-01],
       [8.73835584e-02, 9.12616442e-01],
       [2.43701875e-05, 9.99975630e-01],
       [1.01378968e-06, 9.99998986e-01],
       [5.59886449e-03, 9.94401136e-01],
       [1.55193219e-01, 8.44806781e-01],
       [3.10029329e-01, 6.89970671e-01],
       [1.61254638e-01, 8.38745362e-01],
       [2.15432678e-02, 9.78456732e-01],
       [1.47831134e-05, 9.99985217e-01],
       [2.71572412e-01, 7.28427588e-01],
       [2.11611720e-03, 9.97883883e-01],
       [2.68423368e-03, 9.97315766e-01],
       [1.78996229e-01, 8.21003771e-01],
       [1.19439669e-01, 8.80560331e-01],
       [3.68624113e-07, 9.99999631e-01],
       [2.09801817e-03, 9.97901982e-01],
       [7.40108719e-03, 9.92598913e-01],
       [6.391627

In [18]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [19]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

### Exponentiated Gradient Reduction

In [20]:
eg_red = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                                        constraints='ErrorRateParity', 
                                        eps=0.01, max_iter=50, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

eg_red.fit(X=X_train, y=Y_train)

# Some possible values for constraints: 'DemographicParity', 'EqualizedOdds', 'TruePositiveRateParity', 'ErrorRateParity'

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ExponentiatedGradientReductionEstimator(drop_prot_attr=False,
                                        estimator=LogisticRegression(random_state=123,
                                                                     solver='liblinear'),
                                        prot_attr='age')

In [21]:
Y_test_hat = eg_red.predict(X_test)

In [22]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6047619047619048

In [23]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.2936507936507936

### GridSearchReduction

In [24]:
gs_red = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

In [25]:
gs_red.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

GridSearchReductionEstimator(drop_prot_attr=False,
                             estimator=LogisticRegression(random_state=123,
                                                          solver='liblinear'),
                             prot_attr='age')

In [26]:
Y_test_hat = gs_red.predict(X_test)

In [27]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6428571428571428

In [28]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.3769841269841269

## Post-processing

### PostProcessingMetaEstimator

In [29]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

ppm = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

ppm.fit(X_train, Y_train)

PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                solver='liblinear'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [30]:
Y_test_hat = ppm.predict(X_test)

In [31]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6015873015873016

In [32]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.4206349206349206

In [237]:
roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

ppm = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=roc,
                         prefit=False, val_size=0.25)

ppm.fit(X_train, Y_train)

PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                solver='liblinear'),
                   postprocessor=RejectOptionClassifier(prot_attr='age'))

In [238]:
Y_test_hat = ppm.predict(X_test)

In [239]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6428571428571428

In [240]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.029761904761904767

## Combining processing methods

### Pre + In 

- in pro (ExponentiatedGradientReduction) with pre_pro (ReweighingMetaEstimator) as estimator --> ERROR

In [241]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

In [242]:
in_pro = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=pre_pro, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

in_pro.fit(X=X_train, y=Y_train)
Y_test_hat = in_pro.predict(X_test)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ValueError: Length mismatch: Expected 834 rows, received array of length 850

---

- pre_pro (ReweighingMetaEstimator) with in pro (AdversarialDebiasingEstimator) as estimator --> Same results as using the in pro (AdversarialDebiasingEstimator) alone --> Doesn't work as expected

In [243]:
in_pro = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

ReweighingMetaEstimator(estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                                batch_size=128,
                                                                classifier_num_hidden_units=200,
                                                                debias=True,
                                                                num_epochs=50,
                                                                prot_attr='age',
                                                                random_state=123,
                                                                scope_name='classifier',
                                                                verbose=True),
                        prot_attr='age')

In [244]:
Y_test_hat = pre_pro.predict(X_test)

In [245]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [246]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

---

- pre_pro (ReweighingMetaEstimator) with in pro (ExponentiatedGradientReduction) as estimator --> WORKS! --> different results than when using only the in-pro or only the pre-pro

In [247]:
in_pro = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                                                 constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ReweighingMetaEstimator(estimator=ExponentiatedGradientReductionEstimator(constraints='EqualizedOdds',
                                                                          drop_prot_attr=False,
                                                                          eps=0.01,
                                                                          estimator=LogisticRegression(random_state=123,
                                                                                                       solver='liblinear'),
                                                                          eta0=2.0,
                                                                          max_iter=10,
                                                                          nu=None,
                                                                          prot_attr='age',
                                                                          run_linprog_step=True),
                        prot_attr='age')

In [248]:
Y_test_hat = pre_pro.predict(X_test)

In [249]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6619047619047619

In [250]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.11111111111111116

In [251]:
in_pro.fit(X_train, Y_train)
Y_test_hat = in_pro.predict(X_test)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

In [252]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6507936507936507

In [253]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.1686507936507936

In [254]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)
Y_test_hat = pre_pro.predict(X_test)

In [255]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

In [256]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

---

- pre_pro (ReweighingMetaEstimator) with in pro (GridSearchReduction) as estimator --> Same results as using the in pro (GridSearchReductionEstimator) alone --> Doesn't work as expected

In [257]:
in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ReweighingMetaEstimator(estimator=GridSearchReductionEstimator(constraint_weight=0.5,
                                                               constraints='ErrorRateParity',
                                                               drop_prot_attr=False,
                                                               estimator=LogisticRegression(random_state=123,
                                                                                            solver='liblinear'),
                                                               grid=None,
                                                               grid_limit=2.0,
                                                               grid_size=10,
                                                               loss='ZeroOne',
                                                               max_val=None,
                                                               min_val=None,
                                                               prot_attr='age'),
                        prot_attr='age')

In [258]:
Y_test_hat = pre_pro.predict(X_test)

In [259]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6428571428571428

In [260]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.3769841269841269

In [261]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=pre_pro, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

in_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ValueError: Length mismatch: Expected 1100 rows, received array of length 850

### In + In 

- in pro (ExponentiatedGradientReduction) with in pro (AdversarialDebiasingEstimator) as estimator --> Doestn work as expected --> Same results as when using only in pro (AdversarialDebiasingEstimator)

In [262]:
adversarial_debiasing = AdversarialDebiasingEstimator(   
                                                        prot_attr=sens_variable_name,               
                                                        scope_name='classifier',            
                                                        adversary_loss_weight=0.1,           
                                                        num_epochs=50,                      
                                                        batch_size=128,                     
                                                        classifier_num_hidden_units=200,     
                                                        debias=True,                        
                                                        verbose=True,                     
                                                        random_state=123                   
                                                        )

In [263]:
eg_red = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=adversarial_debiasing, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

eg_red.fit(X=X_train, y=Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

ExponentiatedGradientReductionEstimator(constraints='EqualizedOdds',
                                        drop_prot_attr=False, eps=0.01,
                                        estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                                                batch_size=128,
                                                                                classifier_num_hidden_units=200,
                                                                                debias=True,
                                                                                num_epochs=50,
                                                                                prot_attr='age',
                                                                                random_state=123,
                                                                                scope_name='classifier',
                                                                                verbose=True),
                                        eta0=2.0, max_iter=10, nu=None,
                                        prot_attr='age', run_linprog_step=True)

In [264]:
Y_test_hat = eg_red.predict(X_test)

In [265]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [266]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

### Pre + Post


In [267]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

In [268]:
pre_pro = ReweighingMetaEstimator(estimator=post_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

ReweighingMetaEstimator(estimator=PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                                                  solver='liblinear'),
                                                     postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                                                           random_state=123)),
                        prot_attr='age')

In [269]:
Y_test_hat = pre_pro.predict(X_test)

In [270]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.626984126984127

In [271]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.2579365079365079

---

In [272]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=roc,
                         prefit=False, val_size=0.25)

In [273]:
pre_pro = ReweighingMetaEstimator(estimator=post_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)
Y_test_hat = pre_pro.predict(X_test)

In [274]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6793650793650794

In [275]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.04166666666666674

---

In [276]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=pre_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

PostProcessingMeta(estimator=ReweighingMetaEstimator(estimator=LogisticRegression(random_state=123,
                                                                                  solver='liblinear'),
                                                     prot_attr='age'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [277]:
Y_test_hat = post_pro.predict(X_test)

In [278]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5777777777777777

In [279]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.36111111111111105

---

In [280]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

post_pro = PostProcessingMeta(estimator=pre_pro, postprocessor=roc,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)
Y_test_hat = post_pro.predict(X_test)

In [281]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6238095238095238

In [282]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.1507936507936508

### In + Post

In [283]:
in_pro = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

PostProcessingMeta(estimator=ExponentiatedGradientReductionEstimator(constraints='EqualizedOdds',
                                                                     drop_prot_attr=False,
                                                                     eps=0.01,
                                                                     estimator=LogisticRegression(random_state=123,
                                                                                                  solver='liblinear'),
                                                                     eta0=2.0,
                                                                     max_iter=10,
                                                                     nu=None,
                                                                     prot_attr='age',
                                                                     run_linprog_step=True),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [284]:
Y_test_hat = post_pro.predict(X_test)

In [285]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5349206349206349

In [286]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.25

---

In [287]:
ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                             prefit=False, val_size=0.25)

in_pro = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=post_pro, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

in_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

KeyError: "None of ['age'] are in the columns"

In [ ]:
Y_test_hat = in_pro.predict(X_test)

---

In [289]:
in_pro = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 53.0629; batch adversarial loss: 0.6215
epoch   1; iter:    0; batch classifier loss: 64.5879; batch adversarial loss: 0.5838
epoch   2; iter:    0; batch classifier loss: 59.6540; batch adversarial loss: 0.6089
epoch   3; iter:    0; batch classifier loss: 42.6067; batch adversarial loss: 0.5913
epoch   4; iter:    0; batch classifier loss: 69.6676; batch adversarial loss: 0.6733
epoch   5; iter:    0; batch classifier loss: 38.5424; batch adversarial loss: 0.6096
epoch   6; iter:    0; batch classifier loss: 45.3957; batch adversarial loss: 0.5940
epoch   7; iter:    0; batch classifier loss: 39.3758; batch adversarial loss: 0.6014
epoch   8; iter:    0; batch classifier loss: 34.8769; batch adversarial loss: 0.6264
epoch   9; iter:    0; batch classifier loss: 25.4058; batch adversarial loss: 0.5958
epoch  10; iter:    0; batch classifier loss: 38.2783; batch adversarial loss: 0.5251
epoch  11; iter:    0; batch classifier loss: 52.5652;

PostProcessingMeta(estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                           batch_size=128,
                                                           classifier_num_hidden_units=200,
                                                           debias=True,
                                                           num_epochs=50,
                                                           prot_attr='age',
                                                           random_state=123,
                                                           scope_name='classifier',
                                                           verbose=True),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [290]:
Y_test_hat = post_pro.predict(X_test)

In [291]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5

In [292]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.0

----

In [293]:
in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

PostProcessingMeta(estimator=GridSearchReductionEstimator(constraint_weight=0.5,
                                                          constraints='ErrorRateParity',
                                                          drop_prot_attr=False,
                                                          estimator=LogisticRegression(random_state=123,
                                                                                       solver='liblinear'),
                                                          grid=None,
                                                          grid_limit=2.0,
                                                          grid_size=10,
                                                          loss='ZeroOne',
                                                          max_val=None,
                                                          min_val=None,
                                                          prot_attr='age'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [294]:
Y_test_hat = post_pro.predict(X_test)

In [295]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5476190476190477

In [296]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.6349206349206349

---

In [297]:
ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                             prefit=False, val_size=0.25)

in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=post_pro, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

in_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

KeyError: "None of ['age'] are in the columns"

### Post + Post

In [298]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 


post1_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

post2_pro = PostProcessingMeta(estimator=post1_pro, postprocessor=roc,
                         prefit=False, val_size=0.25)

post2_pro.fit(X_train, Y_train)

TypeError: `estimator` (type: <class 'aif360.sklearn.postprocessing.PostProcessingMeta'>) does not implement method `predict_proba()`.